In [1]:
!pip install pgmpy pygad scikit-learn fairlearn networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 20.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 59.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from fairlearn.metrics import (demographic_parity_difference, 
                               equalized_odds_difference, 
                               true_positive_rate_difference, 
                               false_positive_rate_difference)
from fairlearn.metrics import MetricFrame
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import sparse
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [3]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer


In [4]:
from sklearn.utils import resample
import pandas as pd

#function to upsample in a proportionally biased way

def upsample_minority_groups_german(data, target_col, group_cols, target_value, total_target_samples):
   

    # matches with target to be upsampled
    target_df = data[data[target_col] == target_value]
    
    #groups by sensitive atrributes
    dist = target_df.groupby(group_cols).size().reset_index(name='count')
    total_current_samples = dist['count'].sum()
    
    
    # print(f"Initial {target_col} == {target_value} samples: {total_current_samples}")
    # print("Initial group distribution:")
    # print(dist)

    #creates a table containing all possible combinations of the sensitive attris in the lower bound label
    #on basis of proportion
    
    # Calculate the upsample count for each group
    dist['up_count'] = (dist['count'] * total_target_samples / total_current_samples).round().astype(int)
    # print("Calculated upsample counts for each group:")
    # print(dist)
    
    # Initialize a list to store the upsampled groups
    upsampled_list = []
    
    for _, row in dist.iterrows():
        # it samples for a specific group
        group_condition = True
        for col in group_cols:
            group_condition = group_condition & (target_df[col] == row[col])
        
        #locating the rows having the current group    
        group_df = target_df.loc[group_condition]
        
        #randomly resampling from the group_df
        upsampled_group = resample(group_df, replace=True, n_samples=row['up_count'], random_state=42)
        upsampled_list.append(upsampled_group)
    
    # Concatenate all upsampled groups
    upsampled_df = pd.concat(upsampled_list)
    
    # Combine the upsampled group with the other class
    other_df = data[data[target_col] != target_value]
    final_df = pd.concat([other_df, upsampled_df]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Check the final distribution after upsampling
    # print(f"Final distribution of {target_col} after upsampling:")
    # print(final_df[target_col].value_counts())
    
    return final_df

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def GermanPreprocessing():
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    
    df = pd.read_csv("/kaggle/input/credit-risk-analysis-dataset/german.data", sep=' ', names=column_names)
    sex_map = {
        'A91': 'male', 'A92': 'male', 'A93': 'male',
        'A94': 'female', 'A95': 'female'
    }
    df['sex'] = df['personal_status_sex'].map(sex_map)
    sensitive_sex = df['sex'].map({'male': 0, 'female': 1})
    df['sex'] = sensitive_sex
    
    #categorizing age by setting 0 for <25
    df['age'] = (df['age'] >= 25).astype(int)  

    #A201 is a foreign worker
    
    sensitive_foreign = df['foreign_worker'].map({'A201': 1, 'A202': 0})
    df['foreign_worker'] = sensitive_foreign
    df['credit'] = df['credit'].map({1: 1, 2: 0})
    
    df = df.drop(columns=['personal_status_sex'])
    
    # Upsample the credit=0 class while maintaining sensitive attribute ratios
    df = upsample_minority_groups_german(
        data=df,
        target_col='credit',
        group_cols=['sex', 'age', 'foreign_worker'],
        target_value=0,
        total_target_samples=700 
    )
    
    # print(df['credit'].value_counts())
    # print(df['sex'].value_counts())
    # print(df.groupby(['sex', 'age', 'foreign_worker'])['credit'].value_counts())
    
    
    X = df.drop(columns=['credit'])
    y = df['credit'].values
    
    sensitive_age = df['age'].values
    sensitive_sex = df['sex'].values
    sensitive_foreign = df['foreign_worker'].values
    
    numerical_features = ["duration", "amount", "installment_rate", "residence","number_credits", "people_liable"]
    categorical_features = [col for col in X.columns if col not in numerical_features]
        
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ]
    )
    
    X_train, X_test, y_train, y_test, \
    sensitive_age_train, sensitive_age_test, \
    sensitive_sex_train, sensitive_sex_test, \
    sensitive_foreign_train, sensitive_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=42, stratify=y
    )
    
    pipeline = Pipeline(steps=[("preprocessor", preprocessor)])
    X_train = pipeline.fit_transform(X_train)
    X_test = pipeline.transform(X_test)
    
    return X_train, X_test, y_train, y_test, \
           sensitive_age_train, sensitive_age_test, \
           sensitive_sex_train, sensitive_sex_test, \
           sensitive_foreign_train, sensitive_foreign_test


In [6]:
X_train, X_test, y_train, y_test, sensitive_age_train, sensitive_age_test, sensitive_sex_train, sensitive_sex_test,sensitive_foreign_train,sensitive_foreign_test = GermanPreprocessing()

In [7]:
def four_metrics(y_test, y_pred, sensitive_age_test, sensitive_sex_test,sensitive_foreign_test):
    age_metrics = {
        "Demographic Parity Difference": demographic_parity_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_age_test),
        "Equalized Odds Difference": equalized_odds_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_age_test),
        "Equal Opportunity Difference": true_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_age_test),
        "False Positive Rate Difference": false_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_age_test)
    }
    sex_metrics = {
        "Demographic Parity Difference": demographic_parity_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "Equalized Odds Difference": equalized_odds_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "Equal Opportunity Difference": true_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "False Positive Rate Difference": false_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test)
    }
    foreign_metrics = {
        "Demographic Parity Difference": demographic_parity_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_foreign_test),
        "Equalized Odds Difference": equalized_odds_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_foreign_test),
        "Equal Opportunity Difference": true_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_foreign_test),
        "False Positive Rate Difference": false_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_foreign_test)
    }
    
    return age_metrics, sex_metrics,foreign_metrics

In [8]:
from sklearn.linear_model import LogisticRegression
def base_model():
    print("THIS IS BASE MODEL.")
    base_model = LogisticRegression(max_iter=1000,random_state=42)
    base_model.fit(X_train,y_train)
    base_model_pred= base_model.predict(X_test)
    print("Accuracy", accuracy_score(y_test,base_model_pred))
    # print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    # print("Classification Report:\n", classification_report(y_test, y_pred))
    met_age, met_sex,met_foreign = four_metrics(y_test, base_model_pred, sensitive_age_test, sensitive_sex_test,sensitive_foreign_test)
    print("Fairness metrics for Age:")
    for metric_name, value in met_age.items():
        print(f"{metric_name}: {value:.3f}")
    print("Fairness metrics for Sex:")
    for metric_name, value in met_sex.items():
        print(f"{metric_name}: {value:.3f}")
    print("Fairness metrics for Foreign Indiviual:")  #sus for foreign indiviual
    for metric_name, value in met_foreign.items():
        print(f"{metric_name}: {value:.3f}")
    return base_model_pred
base_model_pred = base_model()

THIS IS BASE MODEL.
Accuracy 0.75
Fairness metrics for Age:
Demographic Parity Difference: 0.152
Equalized Odds Difference: 0.154
Equal Opportunity Difference: 0.154
False Positive Rate Difference: 0.078
Fairness metrics for Sex:
Demographic Parity Difference: 0.075
Equalized Odds Difference: 0.048
Equal Opportunity Difference: 0.015
False Positive Rate Difference: 0.048
Fairness metrics for Foreign Indiviual:
Demographic Parity Difference: 0.511
Equalized Odds Difference: 0.746
Equal Opportunity Difference: 0.258
False Positive Rate Difference: 0.746
